In [2]:

from pathlib import Path
import sys

# Locate the FinWorldModel repository root.
PROJECT_ROOT = Path.cwd().resolve()

while not (PROJECT_ROOT / "pyproject.toml").is_file():
    if PROJECT_ROOT.parent == PROJECT_ROOT:
        raise RuntimeError(
            "Could not locate the FinWorldModel project root "
            "(pyproject.toml not found in the current directory or its parents)."
        )
    PROJECT_ROOT = PROJECT_ROOT.parent

# Add the project's source directory to Python's import path.
SRC_PATH = PROJECT_ROOT / "src"

if not SRC_PATH.is_dir():
    raise RuntimeError(f"Project source directory not found: {SRC_PATH}")

if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

print(f"Project root: {PROJECT_ROOT}")
print(f"Source path:  {SRC_PATH}")


Project root: /home/owner/Projects/FinWorldModel
Source path:  /home/owner/Projects/FinWorldModel/src


In [3]:
from alpaca.data.enums import Adjustment, DataFeed
from alpaca.data.historical import StockHistoricalDataClient
from alpaca.data.models.bars import BarSet
from alpaca.data.requests import StockBarsRequest
from alpaca.data.timeframe import TimeFrame

from fin_data.providers.alpaca.constants import ADJUSTMENTS, BAR_COLUMNS
from fin_data.providers.alpaca.historical import (
    AlpacaHistoricalDataProvider,
    normalize_bars,
)
from fin_data.providers.alpaca.ingestor import default_end_date
from fin_data.universe.instruments import (
    CORPORATE_UNIVERSE,
    ETF_UNIVERSE,
    INITIAL_UNIVERSE,
)
from fin_data.validation.bars import validate_bars

In [4]:
TEST_SYMBOLS = ["SPY", "NVDA"]

universe_by_symbol = {
    instrument.symbol: instrument
    for instrument in INITIAL_UNIVERSE
}

assert len(universe_by_symbol) == len(INITIAL_UNIVERSE), (
    "Duplicate symbols found in INITIAL_UNIVERSE."
)

assert len(ETF_UNIVERSE) == 10
assert len(CORPORATE_UNIVERSE) == 26
assert len(INITIAL_UNIVERSE) == 36

for symbol in TEST_SYMBOLS:
    if symbol not in universe_by_symbol:
        raise ValueError(f"{symbol} is missing from INITIAL_UNIVERSE.")

    print(universe_by_symbol[symbol])

print(f"\nETF count: {len(ETF_UNIVERSE)}")
print(f"Corporate entity count: {len(CORPORATE_UNIVERSE)}")
print(f"Total instruments: {len(INITIAL_UNIVERSE)}")

Instrument(symbol='SPY', asset_class='etf', role='Broad Equity Market', sector=None, industry=None)
Instrument(symbol='NVDA', asset_class='equity', role='Corporate entity', sector='Technology', industry='Semiconductors')

ETF count: 10
Corporate entity count: 26
Total instruments: 36


In [5]:
START_DATE = date(2016, 1, 1)
END_DATE = default_end_date()

if START_DATE > END_DATE:
    raise ValueError(
        f"Invalid date range: {START_DATE} through {END_DATE}"
    )

SAMPLE_START_DATE = max(
    START_DATE,
    END_DATE - timedelta(days=30),
)

print(f"Full history: {START_DATE} through {END_DATE}")
print(f"SDK sample:   {SAMPLE_START_DATE} through {END_DATE}")

Full history: 2016-01-01 through 2026-11-10
SDK sample:   2026-10-11 through 2026-11-10


In [6]:
APCA_API_KEY_ID = os.environ.get("APCA_API_KEY_ID")
APCA_API_SECRET_KEY = os.environ.get("APCA_API_SECRET_KEY")

if not APCA_API_KEY_ID:
    APCA_API_KEY_ID = getpass.getpass("Alpaca API Key ID: ")

if not APCA_API_SECRET_KEY:
    APCA_API_SECRET_KEY = getpass.getpass("Alpaca API Secret Key: ")

if not APCA_API_KEY_ID or not APCA_API_SECRET_KEY:
    raise RuntimeError("Both Alpaca credentials are required.")

stock_client = StockHistoricalDataClient(
    APCA_API_KEY_ID,
    APCA_API_SECRET_KEY,
)

print("Alpaca historical data client initialized.")

Alpaca historical data client initialized.


In [7]:
sample_request = StockBarsRequest(
    symbol_or_symbols="SPY",
    timeframe=TimeFrame.Day,
    start=datetime.combine(
        SAMPLE_START_DATE,
        time.min,
        tzinfo=UTC,
    ),
    end=datetime.combine(
        END_DATE,
        time.max,
        tzinfo=UTC,
    ),
    limit=10_000,
    adjustment=Adjustment.RAW,
    feed=DataFeed.IEX,
)

sample_response = stock_client.get_stock_bars(sample_request)

print(f"Response type: {type(sample_response)}")

if not isinstance(sample_response, BarSet):
    raise TypeError(
        f"Expected BarSet, received {type(sample_response)}"
    )

sample_raw_df = sample_response.df

print(f"DataFrame type: {type(sample_raw_df)}")
print(f"Shape: {sample_raw_df.shape}")
print(f"Index type: {type(sample_raw_df.index)}")
print(f"Index names: {sample_raw_df.index.names}")
print(f"Columns: {sample_raw_df.columns.tolist()}")

display(sample_raw_df.head())
display(sample_raw_df.tail())

Response type: <class 'alpaca.data.models.bars.BarSet'>
DataFrame type: <class 'pandas.core.frame.DataFrame'>
Shape: (22, 7)
Index type: <class 'pandas.core.indexes.multi.MultiIndex'>
Index names: ['symbol', 'timestamp']
Columns: ['open', 'high', 'low', 'close', 'volume', 'trade_count', 'vwap']


open    high      low    close  \
symbol timestamp                                                      
SPY    2026-10-12 04:00:00+00:00  778.740  779.38  777.070  778.030   
       2026-10-13 04:00:00+00:00  777.320  778.75  775.955  778.030   
       2026-10-14 04:00:00+00:00  774.935  776.28  766.710  769.650   
       2026-10-15 04:00:00+00:00  766.270  772.86  764.430  771.770   
       2026-10-16 04:00:00+00:00  773.125  773.18  760.650  763.335   

                                     volume  trade_count        vwap  
symbol timestamp                                                      
SPY    2026-10-12 04:00:00+00:00   727076.0      10797.0  778.025851  
       2026-10-13 04:00:00+00:00   827430.0      13538.0  777.650675  
       2026-10-14 04:00:00+00:00  1498151.0      22723.0  770.184207  
       2026-10-15 04:00:00+00:00  1532510.0      22851.0  769.850664  
       2026-10-16 04:00:00+00:00  1457275.0      22826.0  765.172287

open     high      low   close  \
symbol timestamp                                                      
SPY    2026-11-04 05:00:00+00:00  763.010  769.190  762.710  766.73   
       2026-11-05 05:00:00+00:00  765.540  765.955  732.750  733.43   
       2026-11-06 05:00:00+00:00  734.850  742.110  734.605  735.40   
       2026-11-09 05:00:00+00:00  741.935  755.375  741.180  752.72   
       2026-11-10 05:00:00+00:00  748.310  752.080  744.935  749.93   

                                     volume  trade_count        vwap  
symbol timestamp                                                      
SPY    2026-11-04 05:00:00+00:00  1971119.0      34830.0  766.839277  
       2026-11-05 05:00:00+00:00  3556374.0      55159.0  741.371805  
       2026-11-06 05:00:00+00:00  1941930.0      33217.0  738.151410  
       2026-11-09 05:00:00+00:00  1603276.0      27365.0  751.245552  
       2026-11-10 05:00:00+00:00  1333074.0      25456.0  748.513429

In [8]:
print("DataFrame information:")
sample_raw_df.info()

print("\nColumn data types:")
display(sample_raw_df.dtypes.to_frame("dtype"))

print("\nIndex information:")
print(f"Index type: {type(sample_raw_df.index).__name__}")
print(f"Index names: {sample_raw_df.index.names}")
print(f"Index timezone: {getattr(sample_raw_df.index, 'tz', None)}")

if isinstance(sample_raw_df.index, pd.MultiIndex):
    for level_number, level_name in enumerate(sample_raw_df.index.names):
        print(
            f"Level {level_number}: "
            f"name={level_name!r}, "
            f"values={sample_raw_df.index.get_level_values(level_number).unique()[:5].tolist()}"
        )

DataFrame information:
<class 'pandas.core.frame.DataFrame'>
MultiIndex: 22 entries, ('SPY', Timestamp('2026-10-12 04:00:00+0000', tz='UTC')) to ('SPY', Timestamp('2026-11-10 05:00:00+0000', tz='UTC'))
Data columns (total 7 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   open         22 non-null     float64
 1   high         22 non-null     float64
 2   low          22 non-null     float64
 3   close        22 non-null     float64
 4   volume       22 non-null     float64
 5   trade_count  22 non-null     float64
 6   vwap         22 non-null     float64
dtypes: float64(7)
memory usage: 2.2+ KB

Column data types:


,dtype
open,float64
high,float64
low,float64
close,float64
volume,float64
trade_count,float64
vwap,float64



Index information:
Index type: MultiIndex
Index names: ['symbol', 'timestamp']
Index timezone: None
Level 0: name='symbol', values=['SPY']
Level 1: name='timestamp', values=[Timestamp('2026-10-12 04:00:00+0000', tz='UTC'), Timestamp('2026-10-13 04:00:00+0000', tz='UTC'), Timestamp('2026-10-14 04:00:00+0000', tz='UTC'), Timestamp('2026-10-15 04:00:00+0000', tz='UTC'), Timestamp('2026-10-16 04:00:00+0000', tz='UTC')]


In [9]:
normalized_sample = normalize_bars(
    sample_raw_df,
    "SPY",
)

print(f"Normalized shape: {normalized_sample.shape}")
print(f"Normalized columns: {normalized_sample.columns.tolist()}")
print(f"Index type: {type(normalized_sample.index).__name__}")
print(f"Index timezone: {normalized_sample.index.tz}")

display(normalized_sample.head())
display(normalized_sample.dtypes.to_frame("dtype"))

assert list(normalized_sample.columns) == list(BAR_COLUMNS)
assert isinstance(normalized_sample.index, pd.DatetimeIndex)
assert normalized_sample.index.name == "timestamp"
assert normalized_sample.index.tz is not None
assert normalized_sample.index.is_monotonic_increasing
assert not normalized_sample.index.has_duplicates

print("Normalization checks passed.")

Normalized shape: (22, 7)
Normalized columns: ['open', 'high', 'low', 'close', 'volume', 'trade_count', 'vwap']
Index type: DatetimeIndex
Index timezone: UTC


,open,high,low,close,volume,trade_count,vwap
timestamp,,,,,,,
2026-10-12 04:00:00+00:00,778.740,779.38,777.070,778.030,727076.0,10797.0,778.025851
2026-10-13 04:00:00+00:00,777.320,778.75,775.955,778.030,827430.0,13538.0,777.650675
2026-10-14 04:00:00+00:00,774.935,776.28,766.710,769.650,1498151.0,22723.0,770.184207
2026-10-15 04:00:00+00:00,766.270,772.86,764.430,771.770,1532510.0,22851.0,769.850664
2026-10-16 04:00:00+00:00,773.125,773.18,760.650,763.335,1457275.0,22826.0,765.172287


,dtype
open,float64
high,float64
low,float64
close,float64
volume,float64
trade_count,float64
vwap,float64


Normalization checks passed.


In [10]:
historical_provider = AlpacaHistoricalDataProvider(
    stock_client=stock_client,
    feed=DataFeed.IEX,
)

print(f"Provider: {type(historical_provider).__name__}")
print(f"Feed: {historical_provider.feed}")
print(f"Supported adjustments: {list(ADJUSTMENTS.keys())}")

Provider: AlpacaHistoricalDataProvider
Feed: DataFeed.IEX
Supported adjustments: ['raw', 'all']


In [11]:
historical_data: dict[str, dict[str, pd.DataFrame]] = {}

for symbol in TEST_SYMBOLS:
    historical_data[symbol] = {}

    for adjustment in ("raw", "all"):
        print(
            f"Fetching {symbol}: "
            f"{adjustment} adjustment, "
            f"{START_DATE} through {END_DATE}"
        )

        frame = historical_provider.fetch_daily_bars(
            symbol=symbol,
            start=START_DATE,
            end=END_DATE,
            adjustment=adjustment,
        )

        historical_data[symbol][adjustment] = frame

        print(
            f"Received {len(frame):,} rows "
            f"for {symbol} ({adjustment})."
        )

print("\nHistorical downloads completed.")

Fetching SPY: raw adjustment, 2016-01-01 through 2026-11-10
Received 1,583 rows for SPY (raw).
Fetching SPY: all adjustment, 2016-01-01 through 2026-11-10
Received 1,583 rows for SPY (all).
Fetching NVDA: raw adjustment, 2016-01-01 through 2026-11-10
Received 1,582 rows for NVDA (raw).
Fetching NVDA: all adjustment, 2016-01-01 through 2026-11-10
Received 1,582 rows for NVDA (all).

Historical downloads completed.


In [12]:
validation_results: dict[str, dict[str, list[str]]] = {}

for symbol, adjustments in historical_data.items():
    validation_results[symbol] = {}

    for adjustment, frame in adjustments.items():
        issues = validate_bars(frame, symbol)

        validation_results[symbol][adjustment] = issues

        print(
            f"\n{symbol} / {adjustment}: "
            f"{'PASS' if not issues else 'FAIL'}"
        )

        if issues:
            for issue in issues:
                print(f"  - {issue}")
        else:
            print("  No validation issues.")

all_issues = [
    (symbol, adjustment, issue)
    for symbol, adjustments in validation_results.items()
    for adjustment, issues in adjustments.items()
    for issue in issues
]

if all_issues:
    print(f"\nFound {len(all_issues)} validation issue(s).")
else:
    print("\nAll four datasets passed validation.")


SPY / raw: PASS
  No validation issues.

SPY / all: PASS
  No validation issues.

NVDA / raw: PASS
  No validation issues.

NVDA / all: PASS
  No validation issues.

All four datasets passed validation.


In [13]:
summary_rows = []

for symbol, adjustments in historical_data.items():
    for adjustment, frame in adjustments.items():
        summary_rows.append(
            {
                "symbol": symbol,
                "adjustment": adjustment,
                "rows": len(frame),
                "first_timestamp": (
                    frame.index.min() if not frame.empty else None
                ),
                "last_timestamp": (
                    frame.index.max() if not frame.empty else None
                ),
                "missing_cells": int(frame.isna().sum().sum()),
                "duplicate_timestamps": int(frame.index.duplicated().sum()),
                "monotonic_index": frame.index.is_monotonic_increasing,
                "timezone": str(frame.index.tz),
            }
        )

dataset_summary = pd.DataFrame(summary_rows)

display(dataset_summary)

,symbol,adjustment,rows,first_timestamp,last_timestamp,missing_cells,duplicate_timestamps,monotonic_index,timezone
0,SPY,raw,1583,2018-11-01 04:00:00+00:00,2026-11-10 05:00:00+00:00,0,0,True,UTC
1,SPY,all,1583,2018-11-01 04:00:00+00:00,2026-11-10 05:00:00+00:00,0,0,True,UTC
2,NVDA,raw,1582,2020-07-27 04:00:00+00:00,2026-11-10 05:00:00+00:00,0,0,True,UTC
3,NVDA,all,1582,2020-07-27 04:00:00+00:00,2026-11-10 05:00:00+00:00,0,0,True,UTC


In [14]:
adjustment_comparisons = {}

for symbol in TEST_SYMBOLS:
    raw = historical_data[symbol]["raw"]
    adjusted = historical_data[symbol]["all"]

    comparison = raw[["close"]].join(
        adjusted[["close"]],
        how="inner",
        lsuffix="_raw",
        rsuffix="_all",
    )

    comparison["close_difference"] = (
        comparison["close_all"] - comparison["close_raw"]
    )

    comparison["adjustment_ratio"] = (
        comparison["close_all"] / comparison["close_raw"]
    )

    adjustment_comparisons[symbol] = comparison

    print(f"\n{symbol}")
    print(f"Raw rows: {len(raw):,}")
    print(f"Adjusted rows: {len(adjusted):,}")
    print(f"Matching timestamps: {len(comparison):,}")

    if not comparison.empty:
        display(comparison.head())
        display(comparison.tail())

        print(
            "Adjustment ratio range:",
            comparison["adjustment_ratio"].min(),
            "to",
            comparison["adjustment_ratio"].max(),
        )


SPY
Raw rows: 1,583
Adjusted rows: 1,583
Matching timestamps: 1,583


,close_raw,close_all,close_difference,adjustment_ratio
timestamp,,,,
2018-11-01 04:00:00+00:00,271.60,240.95,-30.65,0.887150
2020-07-27 04:00:00+00:00,323.26,296.94,-26.32,0.918579
2020-07-28 04:00:00+00:00,321.24,295.09,-26.15,0.918597
2020-07-29 04:00:00+00:00,325.16,298.69,-26.47,0.918594
2020-07-30 04:00:00+00:00,323.96,297.58,-26.38,0.918570


,close_raw,close_all,close_difference,adjustment_ratio
timestamp,,,,
2026-11-04 05:00:00+00:00,766.73,766.73,0.0,1.0
2026-11-05 05:00:00+00:00,733.43,733.43,0.0,1.0
2026-11-06 05:00:00+00:00,735.40,735.40,0.0,1.0
2026-11-09 05:00:00+00:00,752.72,752.72,0.0,1.0
2026-11-10 05:00:00+00:00,749.93,749.93,0.0,1.0


Adjustment ratio range: 0.8871502209131074 to 1.0

NVDA
Raw rows: 1,582
Adjusted rows: 1,582
Matching timestamps: 1,582


,close_raw,close_all,close_difference,adjustment_ratio
timestamp,,,,
2020-07-27 04:00:00+00:00,416.955,10.36,-406.595,0.024847
2020-07-28 04:00:00+00:00,409.190,10.17,-399.020,0.024854
2020-07-29 04:00:00+00:00,418.870,10.41,-408.460,0.024853
2020-07-30 04:00:00+00:00,424.260,10.55,-413.710,0.024867
2020-07-31 04:00:00+00:00,423.990,10.54,-413.450,0.024859


,close_raw,close_all,close_difference,adjustment_ratio
timestamp,,,,
2026-11-04 05:00:00+00:00,200.170,200.170,0.0,1.0
2026-11-05 05:00:00+00:00,181.590,181.590,0.0,1.0
2026-11-06 05:00:00+00:00,186.700,186.700,0.0,1.0
2026-11-09 05:00:00+00:00,178.640,178.640,0.0,1.0
2026-11-10 05:00:00+00:00,168.785,168.785,0.0,1.0


Adjustment ratio range: 0.024846523049447487 to 1.0


In [15]:
NVDA_SPLIT_START = pd.Timestamp("2024-06-07", tz="UTC")
NVDA_SPLIT_END = pd.Timestamp("2024-06-14", tz="UTC")

nvda_raw = historical_data["NVDA"]["raw"]
nvda_all = historical_data["NVDA"]["all"]

nvda_split_comparison = pd.DataFrame(
    {
        "raw_close": nvda_raw["close"],
        "adjusted_close": nvda_all["close"],
    }
)

nvda_split_comparison = nvda_split_comparison.loc[
    NVDA_SPLIT_START:NVDA_SPLIT_END
]

nvda_split_comparison["adjustment_ratio"] = (
    nvda_split_comparison["adjusted_close"]
    / nvda_split_comparison["raw_close"]
)

display(nvda_split_comparison)

,raw_close,adjusted_close,adjustment_ratio
timestamp,,,
2024-06-07 04:00:00+00:00,1208.42,120.50,0.099717
2024-06-10 04:00:00+00:00,121.93,121.58,0.997130
2024-06-11 04:00:00+00:00,120.92,120.59,0.997271
2024-06-12 04:00:00+00:00,125.28,124.93,0.997206
2024-06-13 04:00:00+00:00,129.37,129.01,0.997217


In [16]:
for symbol in TEST_SYMBOLS:
    for adjustment in ("raw", "all"):
        frame = historical_data[symbol][adjustment]

        print(f"\n{'=' * 60}")
        print(f"{symbol} / {adjustment}")
        print(f"{'=' * 60}")

        display(frame.describe())


SPY / raw


,open,high,low,close,volume,trade_count,vwap
count,1583.000000,1583.000000,1583.000000,1583.000000,1.583000e+03,1583.000000,1583.000000
mean,508.031772,510.695107,505.128673,508.089084,1.243431e+06,12776.934302,508.016476
std,124.074332,124.280812,123.758770,124.020628,5.701377e+05,6492.858739,124.036881
min,271.600000,271.600000,271.600000,271.600000,2.000000e+02,1.000000,271.600000
25%,411.135000,413.577500,408.510000,411.410000,8.489445e+05,8605.500000,411.179712
50%,457.200000,459.060000,455.120000,456.910000,1.140417e+06,11176.000000,457.034610
75%,598.635000,601.015000,596.380000,598.465000,1.498430e+06,15296.000000,598.829889
max,778.740000,781.590000,777.980000,779.100000,4.791892e+06,55159.000000,779.798150



SPY / all


,open,high,low,close,volume,trade_count,vwap
count,1583.000000,1583.000000,1583.000000,1583.000000,1.583000e+03,1583.000000,1583.000000
mean,491.044842,493.612931,488.245711,491.099968,1.243431e+06,12776.934302,491.030371
std,131.848705,132.120554,131.470250,131.799348,5.701377e+05,6492.858739,131.815101
min,240.950000,240.950000,240.950000,240.950000,2.000000e+02,1.000000,240.950000
25%,387.870000,390.225000,385.830000,387.885000,8.489445e+05,8605.500000,388.200000
50%,434.570000,437.090000,433.300000,434.840000,1.140417e+06,11176.000000,435.360000
75%,587.140000,590.105000,584.740000,587.080000,1.498430e+06,15296.000000,587.380000
max,778.740000,781.590000,777.980000,779.100000,4.791892e+06,55159.000000,779.798150



NVDA / raw


,open,high,low,close,volume,trade_count,vwap
count,1582.000000,1582.000000,1582.000000,1582.000000,1.582000e+03,1582.000000,1582.000000
mean,314.164978,319.446542,308.472370,314.231068,1.801926e+06,20735.082174,314.066174
std,213.539654,216.923231,209.724612,213.705648,1.926095e+06,17735.413438,213.464637
min,87.530000,99.400000,86.620000,94.280000,4.959600e+04,682.000000,95.227483
25%,168.615000,171.263750,164.662500,168.870000,4.355420e+05,7892.750000,168.569547
50%,215.412500,219.525000,211.825000,215.275000,1.038019e+06,15716.500000,215.955710
75%,461.200000,468.852500,452.861250,461.955000,2.788929e+06,29082.500000,461.561475
max,1240.230000,1255.445000,1183.600000,1224.490000,2.522672e+07,198559.000000,1208.531197



NVDA / all


,open,high,low,close,volume,trade_count,vwap
count,1582.000000,1582.000000,1582.000000,1582.000000,1.582000e+03,1582.000000,1582.000000
mean,81.834671,83.101331,80.410196,81.772765,6.538320e+06,20735.082174,81.774927
std,71.958491,72.882356,70.849054,71.822926,4.430797e+06,17735.413438,71.867425
min,10.180000,10.320000,10.150000,10.170000,2.918090e+05,682.000000,10.230000
25%,18.412500,18.927500,18.070000,18.580000,3.326454e+06,7892.750000,18.560000
50%,44.835000,45.680000,44.255000,45.205000,5.089685e+06,15716.500000,45.005000
75%,139.555000,141.442500,137.437500,139.635000,8.754962e+06,29082.500000,139.360000
max,242.040000,243.340000,238.950000,239.170000,3.738908e+07,198559.000000,241.156931


In [17]:
EXPORT_DIR = PROJECT_ROOT / "data" / "processed" / "alpaca_notebook"
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

for symbol, adjustments in historical_data.items():
    for adjustment, frame in adjustments.items():
        csv_path = EXPORT_DIR / f"{symbol}_{adjustment}.csv"

        frame.to_csv(csv_path, index=True)

        print(f"CSV exported: {csv_path}")

        try:
            parquet_path = EXPORT_DIR / f"{symbol}_{adjustment}.parquet"
            frame.to_parquet(parquet_path, index=True)

            print(f"Parquet exported: {parquet_path}")

        except ImportError:
            print(
                "Parquet export skipped: install a Parquet engine "
                "such as pyarrow if you want this format."
            )

CSV exported: /home/owner/Projects/FinWorldModel/data/processed/alpaca_notebook/SPY_raw.csv
Parquet exported: /home/owner/Projects/FinWorldModel/data/processed/alpaca_notebook/SPY_raw.parquet
CSV exported: /home/owner/Projects/FinWorldModel/data/processed/alpaca_notebook/SPY_all.csv
Parquet exported: /home/owner/Projects/FinWorldModel/data/processed/alpaca_notebook/SPY_all.parquet
CSV exported: /home/owner/Projects/FinWorldModel/data/processed/alpaca_notebook/NVDA_raw.csv
Parquet exported: /home/owner/Projects/FinWorldModel/data/processed/alpaca_notebook/NVDA_raw.parquet
CSV exported: /home/owner/Projects/FinWorldModel/data/processed/alpaca_notebook/NVDA_all.csv
Parquet exported: /home/owner/Projects/FinWorldModel/data/processed/alpaca_notebook/NVDA_all.parquet


In [18]:
for symbol in TEST_SYMBOLS:
    for adjustment in ("raw", "all"):
        csv_path = EXPORT_DIR / f"{symbol}_{adjustment}.csv"

        reloaded = pd.read_csv(
            csv_path,
            index_col="timestamp",
            parse_dates=["timestamp"],
        )

        reloaded.index = pd.DatetimeIndex(
            pd.to_datetime(reloaded.index, utc=True),
            name="timestamp",
        )

        original = historical_data[symbol][adjustment]

        assert len(reloaded) == len(original)
        assert list(reloaded.columns) == list(original.columns)
        assert reloaded.index.equals(original.index)

        print(f"Verified: {symbol} / {adjustment}")

Verified: SPY / raw
Verified: SPY / all
Verified: NVDA / raw
Verified: NVDA / all


In [19]:
print("=" * 65)
print("FINWORLD MODEL — ALPACA DATA INSPECTION")
print("=" * 65)

print(f"\nDate range: {START_DATE} through {END_DATE}")
print(f"Feed: {DataFeed.IEX}")
print(f"Timeframe: Daily")
print(f"Instruments: {', '.join(TEST_SYMBOLS)}")

print("\nDataset results:")

for symbol in TEST_SYMBOLS:
    for adjustment in ("raw", "all"):
        frame = historical_data[symbol][adjustment]
        issues = validation_results[symbol][adjustment]

        print(
            f"  {symbol:5} | "
            f"{adjustment:3} | "
            f"{len(frame):6,} rows | "
            f"{'PASS' if not issues else 'FAIL'}"
        )

print("\nValidation:")
print(f"  Total validation issues: {len(all_issues)}")

print("\nChecks performed:")
print("  [x] Direct Alpaca SDK response inspection")
print("  [x] BarSet response type verification")
print("  [x] MultiIndex and timestamp inspection")
print("  [x] Production normalize_bars() execution")
print("  [x] Production historical provider execution")
print("  [x] Raw and fully adjusted historical requests")
print("  [x] Production bar validation")
print("  [x] Dataset coverage and integrity inspection")
print("  [x] Raw versus adjusted comparison")
print("  [x] NVDA split-period inspection")

print("\nNot tested:")
print("  [ ] Corporate actions endpoint and pagination")
print("  [ ] ArcticDB persistence and upsert behavior")
print("  [ ] Ingestion manifest resume and recovery")
print("  [ ] Repeated ingestion idempotence")
print("  [ ] SIP consolidated market data")
print("  [ ] Fundamentals and financial entity relationships")

print("\nExperiment completed.")

FINWORLD MODEL — ALPACA DATA INSPECTION

Date range: 2016-01-01 through 2026-11-10
Feed: DataFeed.IEX
Timeframe: Daily
Instruments: SPY, NVDA

Dataset results:
  SPY   | raw |  1,583 rows | PASS
  SPY   | all |  1,583 rows | PASS
  NVDA  | raw |  1,582 rows | PASS
  NVDA  | all |  1,582 rows | PASS

Validation:
  Total validation issues: 0

Checks performed:
  [x] Direct Alpaca SDK response inspection
  [x] BarSet response type verification
  [x] MultiIndex and timestamp inspection
  [x] Production normalize_bars() execution
  [x] Production historical provider execution
  [x] Raw and fully adjusted historical requests
  [x] Production bar validation
  [x] Dataset coverage and integrity inspection
  [x] Raw versus adjusted comparison
  [x] NVDA split-period inspection

Not tested:
  [ ] Corporate actions endpoint and pagination
  [ ] ArcticDB persistence and upsert behavior
  [ ] Ingestion manifest resume and recovery
  [ ] Repeated ingestion idempotence
  [ ] SIP consolidated market d